# Feature Engineering
from those four files (sessions, users, flights and hotels), I want to create a new file - one row per user with data that may be useable for clustering

In [1]:
#Import
import pandas as pd

In [3]:
#loading data
sessions = pd.read_csv('preprocessed_data/sessions_preprocessed.csv')
users = pd.read_csv('preprocessed_data/users_preprocessed.csv')
hotels = pd.read_csv('preprocessed_data/hotels_preprocessed.csv')
flights = pd.read_csv('preprocessed_data/flights_preprocessed.csv')

In [5]:
#because when trips get cancelled, we have double/dublicated entries for the trip:
#here is a sessions table with only the cancelled ones (to eventually rule out later)
invalid_trip_ids = sessions[sessions["cancellation"]]["trip_id"]

# Users
since a lot of rows can be just taken from this table, I'll start with this one.

In [4]:
users.head()

,user_id,birthdate,gender,married,has_children,home_country,home_city,home_airport,home_airport_lat,home_airport_lon,sign_up_date,Age,days_in
0,94883,1972-03-16,F,True,False,usa,kansas city,MCI,39.297,-94.714,2022-02-07,54,1619
1,101486,1972-12-07,F,True,True,usa,tacoma,TCM,47.138,-122.476,2022-02-17,53,1609
2,101961,1980-09-14,F,True,False,usa,boston,BOS,42.364,-71.005,2022-02-17,45,1609
3,106907,1978-11-17,F,True,True,usa,miami,TNT,25.862,-80.897,2022-02-24,47,1602
4,118043,1972-05-04,F,False,True,usa,los angeles,LAX,33.942,-118.408,2022-03-10,54,1588


In [ ]:
#we already have age and days in.

In [6]:
#age buckets:
def age_group(age):
    if age < 35:
        return "18-35"
    elif age <= 50:
        return "35-50"
    elif age <= 65:
        return "50-65"
    else:
        return "65+"

In [7]:
users["Age Group"] = users["Age"].apply(age_group)

In [8]:
user_features = users[["user_id", "gender", "home_country", "home_city","married", "has_children", "Age Group","days_in"]].copy() # Feature

# Sessions

### how many sessions per trip (before the booking, since the last booking)

In [16]:
sessions['user_id'].value_counts()

user_id
106907    14
252835    14
175032    14
118043    13
101486    13
          ..
551615     8
555075     8
564364     8
575208     8
583955     8
Name: count, Length: 5782, dtype: int64

In [11]:
# test with only one user
sessions_dummy =  sessions[:100][sessions["user_id"] == 217114].copy()

C:\Users\k-kah\AppData\Local\Temp\ipykernel_17416\3168217468.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sessions_dummy =  sessions[:100][sessions["user_id"] == 676570].copy()


In [18]:
#sort sessions for start date
sessions_dummy.sort_values("session_start", inplace=True)

In [ ]:
#backward fill -> all sessions between bookings will get the trip id from the next booked trip
sessions_dummy["trip_id"] = sessions_dummy["trip_id"].bfill()

In [ ]:
# stupidly, with this cancellations seem like bookings. So we need to delete the cancellation sessions

invalid_id = sessions_dummy[sessions_dummy["cancellation"]]["trip_id"]
sessions_dummy_valid = sessions_dummy[~sessions_dummy["trip_id"].isin(invalid_id)]
#aand the average number of session that dummy user needed to book a trip
sessions_dummy_valid.groupby("trip_id")["trip_id"].count().mean()

In [ ]:
#all together for every user:

def session_trip_rate(sessions_dummy):
    sessions_dummy.sort_values("session_start", inplace=True)
    sessions_dummy["trip_id"] = sessions_dummy["trip_id"].bfill()
    invalid_id = sessions_dummy[sessions_dummy["cancellation"]]["trip_id"]
    sessions_dummy_valid = sessions_dummy[~sessions_dummy["trip_id"].isin(invalid_id)]
    return sessions_dummy_valid.groupby("trip_id")["trip_id"].count().mean()

In [ ]:
sesssion_trip_rate_feature = sessions.groupby("user_id").apply(session_trip_rate)  #  Feature

# Hotels